[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshitijrajsharma/deeplearning_scafolding/blob/master/vegetation_segmentation.ipynb)

# Vegetation segmentation: UNet on RGB + NDVI (+ shade)

Tiles are `(6, 250, 250)` uint8 arrays: channels `R, G, B, NDVI, shade, label`.
Label is `0 = background`, `1 = tall`, `2 = flat`.

A from-scratch `smp.Unet` is trained with Lightning, AdamW, kornia flip augmentation,
early stopping on the validation F1, then the best checkpoint is evaluated on the test split.

In [ ]:
# Uncomment on a fresh runtime (e.g. Google Colab) to install dependencies.
# !pip install torch lightning segmentation-models-pytorch torchmetrics kornia pandas geopandas rasterio scikit-learn matplotlib numpy

In [ ]:
import glob
import os

import kornia.augmentation as K
import lightning as L
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio.features
import segmentation_models_pytorch as smp
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.transform import from_origin
from shapely.geometry import shape
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, jaccard_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassF1Score, MulticlassJaccardIndex

CLASS_NAMES = ["background", "tall", "flat"]  # label values 0, 1, 2
RESOLUTION = 0.08  
TILE_CRS = "EPSG:25832"  # ETRS89 / UTM 32N; 
LABEL_CMAP = ListedColormap(["#2b2b2b", "#2ca02c", "#bcbd22"])  # bg, tall, flat

## Config

Flip `USE_SHADE` / `USE_AUG` to toggle the shade channel and augmentation.

In [ ]:
DATA = "data/dataset"
USE_SHADE = True   # True -> 5 input channels (adds shade), False -> 4 (R,G,B,NDVI)
USE_AUG = True     # kornia horizontal/vertical flips during training
EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-3
NUM_WORKERS = 4
PATIENCE = 10
OUT = "outputs"

os.makedirs(OUT, exist_ok=True)
IN_CHANNELS = 5 if USE_SHADE else 4

## Dataset

In [ ]:
class NpyDataset(Dataset):
    """Tiles of shape (6, H, W): R, G, B, NDVI, shade, label."""

    def __init__(self, split_dir, use_shade=True):
        self.files = sorted(glob.glob(os.path.join(split_dir, "*.npy")))
        self.channels = 5 if use_shade else 4

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        tile = np.load(self.files[index]).astype(np.float32)
        image = torch.from_numpy(tile[: self.channels] / 255.0)
        label = torch.from_numpy(tile[5]).long()
        return image, label

## Model

mIoU (`MulticlassJaccardIndex`) and F1 (`MulticlassF1Score`) are the tracked metrics; F1 is the early-stopping target.

In [ ]:
class SegModel(L.LightningModule):
    def __init__(self, in_channels, num_classes=3, lr=1e-3, use_aug=True):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.use_aug = use_aug
        self.net = smp.Unet(
            encoder_name="resnet18",
            encoder_weights=None,
            in_channels=in_channels,
            classes=num_classes,
        )
        self.loss = nn.CrossEntropyLoss()
        self.aug = K.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            data_keys=["input", "mask"],
        )
        self.train_f1 = MulticlassF1Score(num_classes, average="macro")
        self.val_f1 = MulticlassF1Score(num_classes, average="macro")
        self.val_iou = MulticlassJaccardIndex(num_classes, average="macro")
        self.train_history = []
        self.val_history = []

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, _):
        image, label = batch
        if self.use_aug:
            image, mask = self.aug(image, label.unsqueeze(1).float())
            label = mask.squeeze(1).long()
        logits = self(image)
        loss = self.loss(logits, label)
        self.train_f1.update(logits.argmax(1), label)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self.train_history.append(self.train_f1.compute().item())
        self.train_f1.reset()

    def validation_step(self, batch, _):
        image, label = batch
        logits = self(image)
        self.val_f1.update(logits.argmax(1), label)
        self.val_iou.update(logits.argmax(1), label)
        self.log("val_loss", self.loss(logits, label), prog_bar=True)

    def on_validation_epoch_end(self):
        f1 = self.val_f1.compute().item()
        self.log("val_f1", f1, prog_bar=True)
        self.log("val_iou", self.val_iou.compute().item(), prog_bar=True)
        self.val_history.append(f1)
        self.val_f1.reset()
        self.val_iou.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

## Data loaders

In [ ]:
class VegDataModule(L.LightningDataModule):
    def __init__(self, data_dir, use_shade=True, batch_size=16, num_workers=4):
        super().__init__()
        self.data_dir = data_dir
        self.use_shade = use_shade
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.in_channels = 5 if use_shade else 4

    def _loader(self, split, shuffle):
        dataset = NpyDataset(os.path.join(self.data_dir, split), self.use_shade)
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def train_dataloader(self):
        return self._loader("train", shuffle=True)

    def val_dataloader(self):
        return self._loader("val", shuffle=False)

    def test_dataloader(self):
        return self._loader("test", shuffle=False)


data = VegDataModule(DATA, USE_SHADE, BATCH_SIZE, NUM_WORKERS)
test_loader = data.test_dataloader()
data.in_channels

## Class distribution per split

Pixel proportion of each class across train / val / test.

In [ ]:
def proportions(split):
    counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
    for file in glob.glob(os.path.join(DATA, split, "*.npy")):
        label = np.load(file, mmap_mode="r")[5].astype(int).ravel()
        counts += np.bincount(label, minlength=len(CLASS_NAMES))
    return counts / counts.sum()


dist = pd.DataFrame({s: proportions(s) for s in ["train", "val", "test"]}, index=CLASS_NAMES)
dist.T.plot.bar(title="Class proportion per split", ylabel="pixel proportion", rot=0)
plt.show()

## Train

Early stopping watches `val_f1` with patience 10; the best checkpoint is saved.

In [ ]:
model = SegModel(in_channels=data.in_channels, lr=LR, use_aug=USE_AUG)
early_stop = L.pytorch.callbacks.EarlyStopping(monitor="val_f1", mode="max", patience=PATIENCE)
checkpoint = L.pytorch.callbacks.ModelCheckpoint(monitor="val_f1", mode="max", dirpath=OUT, filename="best")
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[early_stop, checkpoint],
    num_sanity_val_steps=0,
    log_every_n_steps=10,
)
trainer.fit(model, datamodule=data)

In [ ]:
optimizer = trainer.optimizers[0]
print(f"optimizer lr: {optimizer.param_groups[0]['lr']}")

## Train vs validation macro F1

In [ ]:
epochs = range(1, len(model.train_history) + 1)
plt.figure()
plt.plot(epochs, model.train_history, marker="o", label="train macro F1")
plt.plot(epochs, model.val_history, marker="o", label="val macro F1")
plt.xlabel("epoch")
plt.ylabel("macro F1")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUT, "f1_curve.png"), bbox_inches="tight")
plt.show()

## Evaluate best checkpoint on test

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
best = SegModel.load_from_checkpoint(checkpoint.best_model_path).eval().to(device)

preds, targets = [], []
with torch.no_grad():
    for image, label in test_loader:
        logits = best(image.to(device))
        preds.append(logits.argmax(1).cpu().numpy().ravel())
        targets.append(label.numpy().ravel())
preds = np.concatenate(preds)
targets = np.concatenate(targets)

labels = list(range(len(CLASS_NAMES)))
print(classification_report(targets, preds, labels=labels, target_names=CLASS_NAMES, digits=4))
print("macro F1 :", round(f1_score(targets, preds, labels=labels, average="macro", zero_division=0), 4))
print("mIoU     :", round(jaccard_score(targets, preds, labels=labels, average="macro", zero_division=0), 4))
per_class_iou = jaccard_score(targets, preds, labels=labels, average=None, zero_division=0)
print("per-class IoU:", dict(zip(CLASS_NAMES, per_class_iou.round(4))))

## Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(targets, preds, display_labels=CLASS_NAMES, normalize="true")
plt.title("Test confusion matrix (row-normalised)")
plt.show()

## Test tiles: image / mask / prediction

In [ ]:
images, masks = next(iter(test_loader))
with torch.no_grad():
    grid_preds = best(images.to(device)).argmax(1).cpu()

n = 3
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for row in range(n):
    rgb = images[row, :3].permute(1, 2, 0).numpy()
    for ax, (data, title) in zip(axes[row], [(rgb, "image"), (masks[row], "mask"), (grid_preds[row], "prediction")]):
        ax.imshow(data, cmap=LABEL_CMAP, vmin=0, vmax=len(CLASS_NAMES) - 1)
        ax.set_title(title)
        ax.axis("off")
handles = [Patch(color=LABEL_CMAP(i), label=name) for i, name in enumerate(CLASS_NAMES)]
fig.legend(handles=handles, loc="lower center", ncol=len(CLASS_NAMES))
plt.show()

## Vectorize predictions to GeoJSON (EPSG:25832)

Tile names encode the SW corner in metres, so each prediction is georeferenced with a 0.08 m/px affine transform, vectorized with `rasterio.features.shapes`, and collected into a geopandas `GeoDataFrame`.

In [ ]:
def tile_polygons(file, prediction):
    name = os.path.splitext(os.path.basename(file))[0]
    east, north = (int(v) for v in name.split("_")[1:3])
    transform = from_origin(east, north + prediction.shape[0] * RESOLUTION, RESOLUTION, RESOLUTION)
    return [
        {"geometry": shape(geom), "class": CLASS_NAMES[int(value)], "tile": name}
        for geom, value in rasterio.features.shapes(prediction.astype("int32"), transform=transform)
        if value != 0
    ]


records = []
for file in test_loader.dataset.files[:6]:
    tile = np.load(file).astype(np.float32)
    image = torch.from_numpy(tile[:IN_CHANNELS] / 255.0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = best(image).argmax(1)[0].cpu().numpy()
    records += tile_polygons(file, pred)

features = gpd.GeoDataFrame(records, crs=TILE_CRS)
features.to_file(os.path.join(OUT, "predictions.geojson"), driver="GeoJSON")
features.plot(column="class", legend=True, cmap="tab10", figsize=(8, 8))
plt.title(f"Vectorized predictions ({TILE_CRS})")
plt.show()
print(len(features), "polygons ->", os.path.join(OUT, "predictions.geojson"))